# 第 8 周解答 —— 最佳性价比房产（Best Value Real Estate）

## 练习目标（领域替换，架构不变）

把「价格竞猜 / The Price is Right」的多 Agent + RAG 流水线换到**房产**领域：找「挂牌价远低于公允估值」的 listing。

架构与第 8 周 Day 4–5 相同：

```
ListingScanner → RAG Estimator（Chroma 可比盘）→ Notify Agent
       ↑                         ↑
   （房源列表）              （相似可比盘 comps）
```

Planning Agent 用 tool loop：扫描 → 逐套估价 → 挑最优 → 通知一次。


## 步骤 1 —— 环境准备（Setup）

导入依赖并做配置。`Deal` / `DealSelection` / `Opportunity` 等数据模型在 `deal_models/deals.py`。


In [39]:
# ========== 导入与全局配置：OpenAI + Chroma + SentenceTransformer ==========

# 标准库：环境变量、路径、JSON、日志
import os
import sys
import json
import logging
from pathlib import Path

# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# OpenAI 官方客户端：后面 chat.completions / tool calling 都用它
from openai import OpenAI
# Chroma：本地持久化向量库，做 RAG 相似检索
import chromadb
# SentenceTransformer：把房源描述编成向量
from sentence_transformers import SentenceTransformer
# 与课程一致的交易/机会模型（本练习复用同一套结构）
from deal_models.deals import Deal, DealSelection, Opportunity

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# 默认读取 OPENAI_API_KEY
openai_client = OpenAI()
# 模型 id 保持原样（影响计费与行为）
MODEL = "gpt-4o-mini"
# 把 INFO 级日志打到控制台，便于观察 tool loop
logging.basicConfig(level=logging.INFO)


## 步骤 2 —— 房产可比盘进 Chroma（RAG 语料）

用一批带描述与价格的样例 listing 建向量库，供后续「估公允价」时做相似度检索（similar comps）。


In [40]:
# ========== 可比盘语料：描述字符串 + 美元价格（保持原样） ==========

# 每条是 (描述, 价格)；描述英文保留，供 embedding 与展示
REAL_ESTATE_COMPS = [
    ("3-bed 2-bath single family home, 1,200 sqft, modern kitchen, hardwood floors, suburban neighborhood", 285_000),
    ("2-bed condo downtown, 900 sqft, balcony, gym, doorman", 420_000),
    ("4-bed house, 2,100 sqft, large yard, garage, good schools", 395_000),
    ("1-bed apartment, 650 sqft, walk-up, near transit", 175_000),
    ("5-bed luxury home, 3,500 sqft, pool, chef's kitchen", 1_200_000),
    ("Studio apartment, 450 sqft, renovated, central location", 155_000),
    ("3-bed townhouse, 1,500 sqft, end unit, small patio", 310_000),
    ("2-bed house, 1,000 sqft, fenced yard, quiet street", 245_000),
    ("4-bed waterfront property, 2,400 sqft, dock", 725_000),
    ("2-bed condo, 1,100 sqft, penthouse, city views", 580_000),
]

# Chroma 持久化目录名（相对工作目录）
DB_PATH = "real_estate_vectorstore"


In [ ]:
# ========== 编码 + 写入 Chroma collection「listings」 ==========

# 轻量句向量模型：与课程常见 MiniLM 一致
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# 拆开描述与价格两列
summaries = [s[0] for s in REAL_ESTATE_COMPS]
prices = [s[1] for s in REAL_ESTATE_COMPS]
# 批量 encode：得到 (N, dim) 向量矩阵
embeddings = encoder.encode(summaries)

# 本地持久化客户端
client = chromadb.PersistentClient(path=DB_PATH)
try:
    # 重跑 notebook 时先删旧集合，避免重复 id
    client.delete_collection("listings")
except Exception:
    pass
# cosine 空间：适合句向量相似度
collection = client.get_or_create_collection("listings", metadata={"hnsw:space": "cosine"})
# ids / embeddings / documents / metadatas 一一对应
collection.add(
    ids=[str(i) for i in range(len(summaries))],
    embeddings=embeddings.tolist(),
    documents=summaries,
    metadatas=[{"price": p} for p in prices],
)
print(f"Chroma ready: {collection.count()} listings")


## 步骤 3 —— ListingScanner 与 RAG Estimator

- **ListingScanner**：返回 Mock 房源（`Deal` 格式），避免真爬网。
- **RAG Estimator**：Chroma 取相似可比盘 → 拼进 prompt → OpenAI 估公允价。


In [42]:
# ========== Mock 扫描：固定 5 套 Deal，JSON 字符串交给 Planning ==========

# 待评估的「市场上的」挂牌（价格/描述/URL 保持原样）
MOCK_LISTINGS = [
    Deal(product_description="2-bed 1-bath condo, 850 sqft, updated kitchen, near parks and transit.", price=198_000, url="https://example.com/listing1"),
    Deal(product_description="3-bed house, 1,300 sqft, garage, backyard. Needs minor updates.", price=265_000, url="https://example.com/listing2"),
    Deal(product_description="1-bed apartment, 600 sqft, new appliances, downtown.", price=142_000, url="https://example.com/listing3"),
    Deal(product_description="4-bed family home, 2,000 sqft, finished basement, great schools.", price=410_000, url="https://example.com/listing4"),
    Deal(product_description="Studio, 450 sqft, loft style, central.", price=128_000, url="https://example.com/listing5"),
]

# Tool 回调：返回 DealSelection 的 JSON 文本（模型读这个再决定估哪几套）
def scan_listings() -> str:
    return DealSelection(deals=MOCK_LISTINGS).model_dump_json()


In [43]:
# ========== RAG 估价：相似可比盘 + LLM 只回数字 ==========

# 从模型回复里抠浮点数
import re

def estimate_fair_value(description: str) -> str:
    # 1) 查询向量：对当前房源描述编码
    vector = encoder.encode([description])
    # 2) Chroma 取 Top-5 相似文档与其 metadata.price
    results = collection.query(query_embeddings=vector.astype(float).tolist(), n_results=5)
    docs = results["documents"][0]
    prices_list = [m["price"] for m in results["metadatas"][0]]
    # 3) 拼可读上下文（截断描述，避免 prompt 过长）
    context = "\n".join([f"Similar: {d[:120]}... | Price: ${p:,.0f}" for d, p in zip(docs, prices_list)])
    # 4) prompt 英文原样：要求只回一个数字
    prompt = f"Estimate the fair market value (USD) for this property. Use similar listings as reference. Respond with only a number.\n\nProperty: {description}\n\nSimilar listings:\n{context}"
    resp = openai_client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
    raw = resp.choices[0].message.content or "0"
    # 去掉千分位逗号后再匹配数字
    match = re.search(r"[-+]?\d*\.?\d+", raw.replace(",", ""))
    val = float(match.group()) if match else 0.0
    # 返回自然语言字符串（Planning 可读；后面 UI 再二次 parse）
    return f"The estimated fair value of this property is ${val:,.2f}"


In [44]:
# ========== 通知工具：本练习用 print 模拟推送 ==========

def notify_user_of_listing(description: str, list_price: float, estimated_value: float, url: str) -> str:
    # 价值缺口 = 估值 - 挂牌价（正数表示「潜在划算」）
    gap = estimated_value - list_price
    print(f"Best Value Alert! List=${list_price:,.0f} | Est=${estimated_value:,.0f} | Value gap=${gap:,.0f}")
    print(f"  {description[:80]}...")
    print(f"  {url}")
    # 给模型一个简短确认，结束 tool 回合
    return "Notification sent (mock)"


## 步骤 4 —— Planning Agent（Tool Loop）

模式与第 4 天相同：声明 tools → 模型决定调用 → 本地执行 → 把结果塞回 messages，循环直到不再 `tool_calls`。


In [45]:
# ========== Tool schema + 本地分发 handle_tool_call ==========

# OpenAI function-calling 格式：name / description / parameters（英文描述给模型看，勿改）
tools = [
    {"type": "function", "function": {"name": "scan_listings", "description": "Returns real estate listings", "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
    {"type": "function", "function": {"name": "estimate_fair_value", "description": "Estimate fair market value from description", "parameters": {"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "notify_user_of_listing", "description": "Notify user of best-value listing; call only once", "parameters": {"type": "object", "properties": {"description": {"type": "string"}, "list_price": {"type": "number"}, "estimated_value": {"type": "number"}, "url": {"type": "string"}}, "required": ["description", "list_price", "estimated_value", "url"], "additionalProperties": False}}},
]

def handle_tool_call(message):
    results = []
    # 一轮里可能有多个 tool_calls，按顺序执行
    for tc in message.tool_calls:
        name = tc.function.name
        # arguments 是 JSON 字符串；空则当 {}
        args = json.loads(tc.function.arguments or "{}")
        # 用 globals 按名字找到同名 Python 函数
        fn = globals().get(name)
        out = fn(**args) if fn else "{}"
        # tool 角色消息必须带上 tool_call_id，才能对齐那次调用
        results.append({"role": "tool", "content": out if isinstance(out, str) else json.dumps(out), "tool_call_id": tc.id})
    return results


In [46]:
# ========== 对话种子：system 定角色，user 给分步指令（英文 prompt 不翻译） ==========

system_msg = "You find great-value real estate listings. Use your tools to scan, estimate each, pick the best value, and notify once. Then reply OK."
user_msg = """First scan for listings. Estimate fair value for each. Pick the single best (list price much lower than estimate). Notify the user of that one, then reply OK."""
# messages 会在 tool loop 里不断 append
messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]


In [ ]:
# ========== Tool loop：直到 finish_reason 不是 tool_calls ==========

done = False
while not done:
    # 把当前 messages + tools 交给模型
    resp = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if resp.choices[0].finish_reason == "tool_calls":
        # 模型要调工具：先把 assistant 消息（含 tool_calls）写回，再追加 tool 结果
        msg = resp.choices[0].message
        results = handle_tool_call(msg)
        messages.append(msg)
        messages.extend(results)
    else:
        # 普通文本回复（期望是 OK）→ 结束循环
        done = True
print("Final reply:", resp.choices[0].message.content)


## 步骤 5 —— Gradio 界面

表格列：Description、List Price、Estimate、Value Gap、URL。点「Surface listings」跑流水线；点某一行触发 `notify_user_of_listing`。


In [ ]:
# ========== Gradio：按钮跑流水线 + 点行通知 ==========

import gradio as gr

def run_pipeline_and_build_table():
    # 扫描 → 解析 DealSelection
    sel = DealSelection.model_validate(json.loads(scan_listings()))
    rows = []
    for d in sel.deals:
        # 复用 RAG 估价；从返回句里再抠数字
        raw = estimate_fair_value(d.product_description)
        m = re.search(r"[-+]?\d*\.?\d+", raw.replace(",", ""))
        est = float(m.group()) if m else 0.0
        # Value Gap：估值高于挂牌的部分（负差夹到 0）
        gap = max(0, round(est - d.price, 2))
        rows.append([d.product_description, d.price, est, gap, d.url])
    return rows

def on_row_select(rows, evt: gr.SelectData):
    # evt.index[0] 是行号；列：0 描述 / 1 挂牌 / 2 估值 / 4 URL
    if evt and rows and 0 <= evt.index[0] < len(rows):
        r = rows[evt.index[0]]
        notify_user_of_listing(r[0], r[1], r[2], r[4])

with gr.Blocks(title="Best Value Real Estate", fill_width=True) as ui:
    gr.Markdown('<div style="text-align: center;font-size:24px">Best Value Real Estate</div>')
    gr.Markdown('<div style="text-align: center;font-size:14px">RAG + multi-agent pipeline</div>')
    tbl = gr.Dataframe(headers=["Description", "List Price", "Estimate", "Value Gap", "URL"], wrap=True, column_widths=[4, 1, 1, 1, 2], row_count=10, col_count=5, max_height=400)
    btn = gr.Button("Surface listings")
    # State 保存当前表格行，供 select 回调读取
    state = gr.State([])
    def on_click():
        rows = run_pipeline_and_build_table()
        # 同时更新 State 与 Dataframe
        return rows, rows
    btn.click(fn=on_click, outputs=[state, tbl])
    tbl.select(fn=on_row_select, inputs=[state], outputs=[])

# 本地启动并尝试打开浏览器
ui.launch(inbrowser=True)
